# Backbone: LLVIP RGB (visible)
YOLOv8n trained on LLVIP visible images, nc=1 (person)

In [1]:
import subprocess, sys
for pkg in ['ultralytics', 'optuna', 'pandas', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Packages OK')

Packages OK


In [2]:
import os, sys, gc, json, glob, shutil, random, math, time
import xml.etree.ElementTree as ET
from collections import defaultdict
import torch
import optuna
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ultralytics import YOLO

optuna.logging.set_verbosity(optuna.logging.WARNING)
BASE_DIR     = '/root/AIP491'
BACKBONE_DIR = os.path.join(BASE_DIR, 'backbones')
DEVICE       = 0
NUM_WORKERS  = 4
IMG_SIZE     = 640
os.makedirs(BACKBONE_DIR, exist_ok=True)
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

torch 2.11.0+cu128 | CUDA: True
GPU: NVIDIA RTX 6000 Ada Generation


In [4]:
LLVIP_DIR     = os.path.join(BASE_DIR, 'data', 'LLVIP')
DATA_DIR      = os.path.join(BASE_DIR, 'data', 'llvip_rgb_yolo')
RUNS_DIR      = os.path.join(BACKBONE_DIR, 'runs', 'llvip_rgb')
BACKBONE_PATH = os.path.join(BACKBONE_DIR, 'llvip_rgb_best.pt')
YAML_PATH     = os.path.join(DATA_DIR, 'dataset.yaml')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

EPOCHS        = 50
PATIENCE      = 5
BATCH         = 16
LR0           = 5e-3
LRF           = 0.01
MOMENTUM      = 0.937
WEIGHT_DECAY  = 5e-4
WARMUP_EPOCHS = 3

print('Backbone: LLVIP RGB -- visible/ images')

Backbone: LLVIP RGB -- visible/ images


In [4]:
# === Chuyen LLVIP VOC -> YOLO format (visible/) ===
def prepare_llvip_yolo(llvip_dir, out_dir, modality='visible'):
    check = os.path.join(out_dir, 'labels', 'train')
    if os.path.isdir(check) and len(os.listdir(check)) > 0:
        print(f'Dataset da ton tai: {out_dir} ({len(os.listdir(check))} train labels)')
        return

    print(f'Dang chuyen LLVIP {modality}/ -> YOLO format...')
    ann_dir = os.path.join(llvip_dir, 'Annotations')
    CLASS_MAP = {'person': 0}

    for split, src_name in [('train', 'train'), ('val', 'test')]:
        img_src = os.path.join(llvip_dir, modality, src_name)
        img_dst = os.path.join(out_dir, 'images', split)
        lbl_dst = os.path.join(out_dir, 'labels', split)
        os.makedirs(img_dst, exist_ok=True)
        os.makedirs(lbl_dst, exist_ok=True)

        imgs = sorted([f for f in os.listdir(img_src)
                       if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        count = 0
        for fname in imgs:
            stem = os.path.splitext(fname)[0]
            xml_path = os.path.join(ann_dir, stem + '.xml')
            if not os.path.exists(xml_path):
                continue
            tree = ET.parse(xml_path)
            root = tree.getroot()
            sz = root.find('size')
            w, h = int(sz.find('width').text), int(sz.find('height').text)

            lines = []
            for obj in root.findall('object'):
                name = obj.find('name').text.strip().lower()
                if name not in CLASS_MAP:
                    continue
                bnd = obj.find('bndbox')
                xmin = max(0.0, float(bnd.find('xmin').text))
                ymin = max(0.0, float(bnd.find('ymin').text))
                xmax = min(float(w), float(bnd.find('xmax').text))
                ymax = min(float(h), float(bnd.find('ymax').text))
                if xmax <= xmin or ymax <= ymin:
                    continue
                cx = (xmin + xmax) / 2 / w
                cy = (ymin + ymax) / 2 / h
                bw = (xmax - xmin) / w
                bh = (ymax - ymin) / h
                lines.append(f'{CLASS_MAP[name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

            if not lines:
                continue
            shutil.copy2(os.path.join(img_src, fname), os.path.join(img_dst, fname))
            with open(os.path.join(lbl_dst, stem + '.txt'), 'w') as f:
                f.write('\n'.join(lines))
            count += 1
        print(f'  {split}: {count} samples')

prepare_llvip_yolo(LLVIP_DIR, DATA_DIR, modality='visible')

with open(YAML_PATH, 'w') as f:
    f.write(f'path: {DATA_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [person]\n')
print(f'YAML: {YAML_PATH}')

Dang chuyen LLVIP visible/ -> YOLO format...
  train: 12023 samples
  val: 3463 samples
YAML: /root/AIP491/data/llvip_rgb_yolo/dataset.yaml


In [5]:
# === Train final model ===
if os.path.exists(BACKBONE_PATH):
    print(f'Backbone da ton tai: {BACKBONE_PATH}')
else:
    print('Training...')
    model = YOLO('yolov8n.pt')
    results = model.train(
        data=YAML_PATH,
        epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
        lr0=LR0, lrf=LRF, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS, device=DEVICE, workers=NUM_WORKERS,
        patience=PATIENCE, project=RUNS_DIR, name='final',
        exist_ok=True, verbose=True, single_cls=True,
        fliplr=0.5, mosaic=1.0, scale=0.5,
    )
    del model; torch.cuda.empty_cache(); gc.collect()
    print('Training done.')

Training...
Ultralytics 8.4.41 🚀 Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48520MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/AIP491/data/llvip_rgb_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final, nbs=64, nms=False, opset=None, optimize=False, o

In [6]:
# === Luu backbone va danh gia ===
best_pt = os.path.join(RUNS_DIR, 'final', 'weights', 'best.pt')
if os.path.exists(best_pt):
    shutil.copy2(best_pt, BACKBONE_PATH)
    print(f'Backbone saved: {BACKBONE_PATH}')
else:
    print('ERROR: best.pt not found, check training.')

# Evaluation
if os.path.exists(BACKBONE_PATH):
    model = YOLO(BACKBONE_PATH)
    metrics = model.val(data=YAML_PATH, imgsz=IMG_SIZE, batch=BATCH,
                        workers=NUM_WORKERS, verbose=False)
    m = metrics.results_dict
    print(f'mAP@0.5:      {m.get("metrics/mAP50(B)", 0):.4f}')
    print(f'mAP@0.5:0.95: {m.get("metrics/mAP50-95(B)", 0):.4f}')
    print(f'Precision:    {m.get("metrics/precision(B)", 0):.4f}')
    print(f'Recall:       {m.get("metrics/recall(B)", 0):.4f}')
    del model; torch.cuda.empty_cache(); gc.collect()

# Loss curve
csv_path = os.path.join(RUNS_DIR, 'final', 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    fig, ax = plt.subplots(figsize=(10, 4))
    if 'train/box_loss' in df.columns:
        train_loss = (df['train/box_loss']
                      + df.get('train/cls_loss', 0)
                      + df.get('train/dfl_loss', 0))
        val_loss   = (df['val/box_loss']
                      + df.get('val/cls_loss', 0)
                      + df.get('val/dfl_loss', 0))
        ep = range(1, len(train_loss) + 1)
        ax.plot(ep, train_loss, label='Train Loss')
        ax.plot(ep, val_loss, '--', label='Val Loss')
    ax.set(title='LLVIP RGB Backbone -- Loss', xlabel='Epoch', ylabel='Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = os.path.join(RUNS_DIR, 'loss_curves.png')
    plt.savefig(save_path, dpi=150); plt.show()
    print(f'Loss curves: {save_path}')

Backbone saved: /root/AIP491/backbones/llvip_rgb_best.pt
Ultralytics 8.4.41 🚀 Python-3.10.20 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48520MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 653.6±142.3 MB/s, size: 101.1 KB)
val: Scanning /root/AIP491/data/llvip_rgb_yolo/labels/val.cache... 3463 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3463/3463 354.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 217/217 5.2it/s 41.6s<0.1s
                   all       3463       8302      0.891      0.817      0.882       0.51
Speed: 1.7ms preprocess, 2.0ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /root/AIP491/runs/detect/val
mAP@0.5:      0.8820
mAP@0.5:0.95: 0.5100
Precision:    0.8913
Recall:       0.8174
Loss curves: /root/AIP491/backbones/runs/llvip_rgb/loss_curves.png
